In [1]:
import pandas as pd
import importlib
from scripts.preprocessing import get_data
from scripts.paths import RAW_XLSX

In [2]:
df = get_data(RAW_XLSX)
df

,payment_id,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success,age,city,device_type
0,5001,1001.0,A,2024-04-16 06:06:00,2024-04-16 06:10:00,NaT,NaT,31,Москва,ios
1,5002,1002.0,A,2024-04-12 07:38:00,2024-04-12 07:41:00,NaT,NaT,31,Москва,ios
2,5003,1003.0,A,2024-04-26 20:03:00,2024-04-26 20:07:00,NaT,NaT,25,Москва,android
3,5004,1004.0,A,2024-04-09 16:02:00,NaT,NaT,NaT,20,Москва,ios
4,5005,1004.0,A,2024-04-04 03:46:00,2024-04-04 03:54:00,2024-04-04 03:57:00,2024-04-04 04:02:00,20,Москва,ios
...,...,...,...,...,...,...,...,...,...,...
2654,7655,2499.0,B,2024-04-14 19:30:00,2024-04-14 19:31:00,2024-04-14 19:38:00,2024-04-14 19:44:00,42,Москва,ios
2655,7656,2499.0,B,2024-04-24 20:54:00,2024-04-24 21:00:00,2024-04-24 21:03:00,2024-04-24 21:08:00,42,Москва,ios
2656,7657,2500.0,B,2024-04-11 07:01:00,2024-04-11 07:04:00,NaT,NaT,39,Москва,ios
2657,7658,9004.0,A,2024-04-05 16:14:00,2024-04-05 16:15:00,2024-04-05 16:18:00,2024-04-05 16:23:00,30,NaN,android


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2659 entries, 0 to 2658
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   payment_id       2659 non-null   int64         
 1   user_id          2659 non-null   float64       
 2   group            2659 non-null   str           
 3   step1_opened     2659 non-null   datetime64[us]
 4   step2_entered    2255 non-null   datetime64[us]
 5   step3_confirmed  1718 non-null   datetime64[us]
 6   step4_success    1521 non-null   datetime64[us]
 7   age              2659 non-null   int64         
 8   city             2658 non-null   str           
 9   device_type      2659 non-null   str           
dtypes: datetime64[us](4), float64(1), int64(2), str(3)
memory usage: 207.9 KB


In [4]:
step_cols = ["step1_opened", "step2_entered", "step3_confirmed", "step4_success"]

df = df.copy()
df[step_cols] = df[step_cols].notna().astype(int)

users = (
    df.drop(columns=["payment_id"])
      .groupby("user_id", as_index=False)
      .agg({
          "group" : "first",
          **{c: "sum" for c in step_cols},
          "age": "first",
          "city": "first",
          "device_type": "first",
      })
)
users

,user_id,group,step1_opened,step2_entered,step3_confirmed,step4_success,age,city,device_type
0,1001.0,A,1,1,0,0,31,Москва,ios
1,1002.0,A,1,1,0,0,31,Москва,ios
2,1003.0,A,1,1,0,0,25,Москва,android
3,1004.0,A,3,2,2,2,20,Москва,ios
4,1005.0,A,2,2,1,1,34,Москва,ios
...,...,...,...,...,...,...,...,...,...
1496,2497.0,B,1,0,0,0,33,Москва,android
1497,2498.0,B,1,1,1,1,28,СПб,ios
1498,2499.0,B,3,3,3,3,42,Москва,ios
1499,2500.0,B,1,1,0,0,39,Москва,ios


In [5]:
users.step1_opened.value_counts()

step1_opened
1    741
2    442
3    238
4     80
Name: count, dtype: int64

In [6]:
import numpy as np
from scipy.stats import chisquare, norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    confint_proportions_2indep,
    proportion_effectsize,
    proportions_ztest,
)

ALPHA, POWER = 0.05, 0.8
z_alpha = norm.ppf(1 - ALPHA / 2)
z_beta = norm.ppf(POWER) 

opened = users[users.step1_opened > 0]
conv = (opened.step4_success > 0).groupby(opened.group).agg(["sum", "count"])
x_a, n_a = (int(v) for v in conv.loc["A"])
x_b, n_b = (int(v) for v in conv.loc["B"])

srm_p = chisquare([n_a, n_b]).pvalue

p_a, p_b = x_a / n_a, x_b / n_b
diff = p_b - p_a

p_pool = (x_a + x_b) / (n_a + n_b)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n_a + 1 / n_b))
z = diff / se_pool
p_value = 2 * (1 - norm.cdf(abs(z)))

se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci_low, ci_high = diff - z_alpha * se, diff + z_alpha * se

z_sm, p_sm = proportions_ztest([x_b, x_a], [n_b, n_a])
ci_sm = confint_proportions_2indep(x_b, n_b, x_a, n_a, method="wald", compare="diff")
assert np.isclose(z, z_sm) and np.isclose(p_value, p_sm)
assert np.allclose((ci_low, ci_high), ci_sm)

mde = (z_alpha + z_beta) * np.sqrt(p_a * (1 - p_a) * (1 / n_a + 1 / n_b))
h = NormalIndPower().solve_power(nobs1=n_a, ratio=n_b / n_a, alpha=ALPHA, power=POWER)
mde_sm = np.sin(np.arcsin(np.sqrt(p_a)) + h / 2) ** 2 - p_a

power_obs = NormalIndPower().power(
    effect_size=proportion_effectsize(p_b, p_a), nobs1=n_a, ratio=n_b / n_a, alpha=ALPHA
)

summary = pd.Series(
    {
        "n_A / n_B": f"{n_a} / {n_b}",
        "SRM p-value": f"{srm_p:.3f}",
        "Конверсия A": f"{p_a:.2%} ({x_a}/{n_a})",
        "Конверсия B": f"{p_b:.2%} ({x_b}/{n_b})",
        "Разница B-A": f"{diff * 100:+.2f} п.п.",
        "Относительный uplift": f"{diff / p_a:+.1%}",
        "95% ДИ разницы": f"[{ci_low * 100:+.2f}; {ci_high * 100:+.2f}] п.п.",
        "SE": f"{se:.4f}",
        "z": f"{z:.3f}",
        "p-value": f"{p_value:.2e}",
        f"MDE (alpha={ALPHA}, power={POWER})": f"{mde * 100:.2f} п.п. (statsmodels: {mde_sm * 100:.2f})",
        "Мощность для наблюдаемого эффекта": f"{power_obs:.3f}",
    },
    name="Конверсия 1->4, уровень клиента",
)
summary

n_A / n_B                                                751 / 750
SRM p-value                                                  0.979
Конверсия A                                       65.65% (493/751)
Конверсия B                                       80.40% (603/750)
Разница B-A                                            +14.75 п.п.
Относительный uplift                                        +22.5%
95% ДИ разницы                               [+10.33; +19.18] п.п.
SE                                                          0.0226
z                                                            6.439
p-value                                                   1.20e-10
MDE (alpha=0.05, power=0.8)          6.87 п.п. (statsmodels: 6.68)
Мощность для наблюдаемого эффекта                            1.000
Name: Конверсия 1->4, уровень клиента, dtype: str

In [28]:
from statsmodels.stats.multitest import multipletests

reached = (opened[step_cols] > 0).groupby(opened.group).sum()
step_names = ["1. Открыл форму", "2. Ввёл реквизиты", "3. Подтвердил", "4. Успешная оплата"]

rows = []
for i, (col, name) in enumerate(zip(step_cols, step_names, strict=True)):
    prev = step_cols[max(i - 1, 0)]
    k_a, k_b = int(reached.at["A", col]), int(reached.at["B", col])
    m_a, m_b = int(reached.at["A", prev]), int(reached.at["B", prev])
    from_start_a, from_start_b = k_a / n_a, k_b / n_b  
    step_a, step_b = k_a / m_a, k_b / m_b  
    p = proportions_ztest([k_b, k_a], [m_b, m_a])[1] if i > 0 else np.nan
    rows.append(
        {
            "Шаг": name,
            "A, от шага 1": from_start_a,
            "B, от шага 1": from_start_b,
            "A, из пред. шага": step_a,
            "B, из пред. шага": step_b,
            "Δ абс., п.п.": (step_b - step_a) * 100,
            "Δ отн.": step_b / step_a - 1,
            "p-value": p,
        }
    )

funnel = pd.DataFrame(rows).set_index("Шаг")

funnel.loc[funnel.index[1:], "p-value (Holm)"] = multipletests(
    funnel["p-value"].iloc[1:], method="holm"
)[1]

pct = "{:.1%}".format
formats = {
    "A, от шага 1": pct,
    "B, от шага 1": pct,
    "A, из пред. шага": pct,
    "B, из пред. шага": pct,
    "Δ абс., п.п.": "{:+.2f}".format,
    "Δ отн.": "{:+.1%}".format,
    "p-value": "{:.2e}".format,
    "p-value (Holm)": "{:.2e}".format,
}
funnel_view = funnel.copy().astype(object)
for col, fmt in formats.items():
    funnel_view[col] = funnel[col].map(lambda v, f=fmt: "-" if pd.isna(v) else f(v))
funnel_view

,"A, от шага 1","B, от шага 1","A, из пред. шага","B, из пред. шага","Δ абс., п.п.",Δ отн.,p-value,p-value (Holm)
Шаг,,,,,,,,
1. Открыл форму,100.0%,100.0%,100.0%,100.0%,+0.00,+0.0%,-,-
2. Ввёл реквизиты,89.3%,94.3%,89.3%,94.3%,+4.92,+5.5%,5.12e-04,1.02e-03
3. Подтвердил,73.0%,85.9%,81.7%,91.1%,+9.42,+11.5%,3.14e-07,9.41e-07
4. Успешная оплата,65.6%,80.4%,90.0%,93.6%,+3.67,+4.1%,2.03e-02,2.03e-02


In [29]:
def segment_effect(data: pd.DataFrame, segment: str) -> pd.DataFrame:
    """Эффект B vs A по конверсии 1->4 (уровень клиента) внутри каждого значения сегмента."""
    rows = []
    for value, g in data.dropna(subset=[segment]).groupby(segment):
        t = (g.step4_success > 0).groupby(g.group).agg(["sum", "count"])
        sx_a, sn_a = (int(v) for v in t.loc["A"])
        sx_b, sn_b = (int(v) for v in t.loc["B"])
        sp_a, sp_b = sx_a / sn_a, sx_b / sn_b
        _, sp = proportions_ztest([sx_b, sx_a], [sn_b, sn_a])
        lo, hi = confint_proportions_2indep(sx_b, sn_b, sx_a, sn_a, method="wald", compare="diff")
        rows.append(
            {
                segment: value,
                "n_A / n_B": f"{sn_a} / {sn_b}",
                "Конверсия A": f"{sp_a:.1%}",
                "Конверсия B": f"{sp_b:.1%}",
                "Δ абс., п.п.": f"{(sp_b - sp_a) * 100:+.2f}",
                "Δ отн.": f"{sp_b / sp_a - 1:+.1%}",
                "95% ДИ, п.п.": f"[{lo * 100:+.1f}; {hi * 100:+.1f}]",
                "p-value": sp,
            }
        )
    res = pd.DataFrame(rows).set_index(segment)
    # Поправка Холма на число сравнений внутри разреза
    res["p-value (Holm)"] = multipletests(res["p-value"], method="holm")[1]
    for col in ["p-value", "p-value (Holm)"]:
        res[col] = res[col].map("{:.2e}".format)
    return res


segment_effect(opened, "device_type")

,n_A / n_B,Конверсия A,Конверсия B,"Δ абс., п.п.",Δ отн.,"95% ДИ, п.п.",p-value,p-value (Holm)
device_type,,,,,,,,
android,407 / 369,64.4%,79.9%,+15.57,+24.2%,[+9.4; +21.8],1.48e-06,2.97e-06
ios,344 / 381,67.2%,80.8%,+13.69,+20.4%,[+7.3; +20.0],2.51e-05,2.51e-05


In [30]:
segment_effect(opened, "city")

,n_A / n_B,Конверсия A,Конверсия B,"Δ абс., п.п.",Δ отн.,"95% ДИ, п.п.",p-value,p-value (Holm)
city,,,,,,,,
Москва,288 / 306,62.5%,81.0%,+18.55,+29.7%,[+11.4; +25.7],4.80e-07,1.44e-06
Регион,268 / 245,66.8%,76.3%,+9.54,+14.3%,[+1.8; +17.3],1.70e-02,1.70e-02
СПб,194 / 199,68.6%,84.4%,+15.87,+23.1%,[+7.6; +24.1],2.04e-04,4.09e-04
